# Model Development and Evaluation

This notebook develops and evaluates forecasting models for daily retail demand at the store-category level.

The modeling workflow uses chronological train, validation, and test periods to preserve the temporal structure of the forecasting problem. A 28-day forecast horizon is used, and target-derived predictors are restricted to information available at the forecast origin to prevent leakage.

The workflow includes:
- Time-based train, validation, and test splitting
- Seasonal and rolling-average forecasting baselines
- Leakage-safe feature selection
- Random Forest and gradient-boosting models
- Model comparison using WAPE, MAE, RMSE, and forecast bias
- Final evaluation on an untouched 28-day test period

In [3]:
from pathlib import Path
import sys

import pandas as pd

# Add the project root to the Python path so modules under src/ can be imported.
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_m5_data
from src.data.preprocess import preprocess_m5_data
from src.data.split_data import (
    time_series_split,
    validate_time_series_split,
)
from src.features.build_features import build_features
from src.models.model_data import prepare_ml_data
from src.models.baselines import (
    seasonal_naive_forecast,
    rolling_mean_forecast,
)
from src.evaluation.metrics import (
    mae,
    rmse,
    wape,
    bias_pct,
)

print("Project root:", PROJECT_ROOT)

Project root: c:\Data Science\GitHub\retail-demand-forecasting


## 1. Data Preparation

The raw M5 datasets are loaded and passed through the reusable preprocessing and feature-engineering pipelines defined under `src/`.

In [7]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Load the raw M5 datasets.
sales, calendar, prices = load_m5_data(DATA_DIR)

# Transform the item-level data into store-category daily observations.
data = preprocess_m5_data(
    sales,
    calendar,
    prices,
)

# Generate the complete forecasting feature dataset.
model_data = build_features(data)

print("Preprocessed data:", data.shape)
print("Modeling data:", model_data.shape)
print(
    "Date range:",
    model_data["date"].min().date(),
    "to",
    model_data["date"].max().date(),
)

Preprocessed data: (58230, 19)
Modeling data: (56550, 62)
Date range: 2011-03-26 to 2016-05-22


## 2. Time-Based Data Split

The final 28 days are reserved as an untouched test set. The preceding 28 days form the validation set used for model comparison and selection. All earlier observations are used for training.

Unlike a random split, this structure ensures that models are evaluated on future periods relative to their training data.

In [8]:
train, validation, test = time_series_split(
    model_data,
    validation_days=28,
    test_days=28,
)

validate_time_series_split(
    train,
    validation,
    test,
)

for name, dataset in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test),
]:
    print(
        f"{name}: "
        f"{dataset['date'].min().date()} to "
        f"{dataset['date'].max().date()} | "
        f"{len(dataset):,} rows | "
        f"{dataset['date'].nunique()} days"
    )

Train: 2011-03-26 to 2016-03-27 | 54,870 rows | 1829 days
Validation: 2016-03-28 to 2016-04-24 | 840 rows | 28 days
Test: 2016-04-25 to 2016-05-22 | 840 rows | 28 days


## 3. Baseline Forecasts

Simple forecasting methods provide reference points that machine-learning models should outperform.

Two baselines are evaluated:
- **7-day seasonal naïve:** uses demand from the same weekday one week earlier.
- **28-day rolling mean:** uses recent average demand.

Baseline performance is measured on the validation period using MAE, RMSE, WAPE, and forecast bias.

In [9]:
baseline_predictions = {
    "Seasonal Naive (7-day)": seasonal_naive_forecast(
        validation,
        lag_days=7,
    ),
    "Rolling Mean (28-day)": rolling_mean_forecast(
        validation,
        window=28,
    ),
}

baseline_results = []

for model_name, predictions in baseline_predictions.items():
    baseline_results.append(
        {
            "Model": model_name,
            "MAE": mae(validation["sales"], predictions),
            "RMSE": rmse(validation["sales"], predictions),
            "WAPE (%)": wape(validation["sales"], predictions),
            "Bias (%)": bias_pct(validation["sales"], predictions),
        }
    )

baseline_results = (
    pd.DataFrame(baseline_results)
    .set_index("Model")
)

baseline_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Seasonal Naive (7-day),170.05,298.87,12.07,0.18
Rolling Mean (28-day),232.46,364.12,16.50,-0.69


## 4. Leakage-Safe Modeling Features

For a 28-day batch forecast, target-derived features must be available at the beginning of the forecast horizon.

Therefore, short target lags such as 1, 7, 14, and 21 days are excluded from the ML feature set. Demand lags of 28, 35, 42, and 56 days remain available for every date in the forecast horizon.

Calendar, event, SNAP, and price variables are treated as known future covariates.

In [10]:
X_train, y_train = prepare_ml_data(train)
X_validation, y_validation = prepare_ml_data(validation)

print("Training predictors:", X_train.shape)
print("Training target:", y_train.shape)

print("Validation predictors:", X_validation.shape)
print("Validation target:", y_validation.shape)

print(
    "Missing predictor values:",
    X_train.isna().sum().sum()
    + X_validation.isna().sum().sum(),
)

print(
    "Target included in predictors:",
    "sales" in X_train.columns,
)

Training predictors: (54870, 30)
Training target: (54870,)
Validation predictors: (840, 30)
Validation target: (840,)
Missing predictor values: 0
Target included in predictors: False


## 5. Random Forest

Random Forest provides the first machine-learning benchmark. It can capture nonlinear relationships and interactions among demand history, calendar, price, event, and store-category features without requiring strong assumptions about the functional form.

Categorical variables are one-hot encoded as part of a scikit-learn pipeline, ensuring that preprocessing is learned from the training data and applied consistently during prediction.

The initial model is intentionally not tuned. Its purpose is to establish baseline machine-learning performance before hyperparameter optimization.

In [11]:
from src.models.train_random_forest import (
    build_random_forest_pipeline,
)

rf_pipeline = build_random_forest_pipeline(
    n_estimators=300,
    random_state=42,
)

rf_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
# Fit preprocessing and the Random Forest model using training data only.
rf_pipeline.fit(
    X_train,
    y_train,
)

print("Random Forest training complete.")

Random Forest training complete.


In [13]:
rf_validation_pred = rf_pipeline.predict(
    X_validation
)

print("Predictions:", len(rf_validation_pred))
print(
    "Prediction range:",
    f"{rf_validation_pred.min():.2f}",
    "to",
    f"{rf_validation_pred.max():.2f}",
)

Predictions: 840
Prediction range: 227.09 to 4938.54


In [14]:
rf_results = pd.DataFrame(
    {
        "Model": ["Random Forest"],
        "MAE": [
            mae(
                y_validation,
                rf_validation_pred,
            )
        ],
        "RMSE": [
            rmse(
                y_validation,
                rf_validation_pred,
            )
        ],
        "WAPE (%)": [
            wape(
                y_validation,
                rf_validation_pred,
            )
        ],
        "Bias (%)": [
            bias_pct(
                y_validation,
                rf_validation_pred,
            )
        ],
    }
).set_index("Model")

rf_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Random Forest,111.52,178.77,7.91,0.5


In [15]:
validation_results = pd.concat(
    [
        baseline_results,
        rf_results,
    ]
)

validation_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Seasonal Naive (7-day),170.05,298.87,12.07,0.18
Rolling Mean (28-day),232.46,364.12,16.50,-0.69
Random Forest,111.52,178.77,7.91,0.50


### Random Forest Validation Results

The untuned Random Forest achieved a validation WAPE of **7.91%**, with an MAE of **111.52** and RMSE of **178.77**. Aggregate forecast bias was **+0.50%**, indicating limited systematic overforecasting.

The model performs strongly despite restricting target-derived predictors to lags of at least 28 days, ensuring that demand information from within the 28-day validation horizon is not used as an input.

The 7-day seasonal-naïve benchmark achieved a higher WAPE of 12.07%. This benchmark represents a rolling daily forecast because it uses actual demand from seven days earlier, including observations within the validation horizon, whereas the Random Forest is evaluated as a 28-day batch forecast from a single forecast origin.

In [16]:
try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost version: 2.0.3


## 6. XGBoost

XGBoost is evaluated as a gradient-boosted tree model capable of capturing nonlinear relationships and feature interactions in the demand data.

The model uses the same leakage-safe feature set and categorical preprocessing as Random Forest, enabling a consistent comparison between the two approaches.

The initial configuration is evaluated without hyperparameter tuning.

In [17]:
from src.models.train_xgboost import (
    build_xgboost_pipeline,
)

xgb_pipeline = build_xgboost_pipeline(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

xgb_pipeline.fit(
    X_train,
    y_train,
)

print("XGBoost training complete.")

XGBoost training complete.


In [18]:
xgb_validation_pred = xgb_pipeline.predict(
    X_validation
)

xgb_results = pd.DataFrame(
    {
        "Model": ["XGBoost"],
        "MAE": [
            mae(
                y_validation,
                xgb_validation_pred,
            )
        ],
        "RMSE": [
            rmse(
                y_validation,
                xgb_validation_pred,
            )
        ],
        "WAPE (%)": [
            wape(
                y_validation,
                xgb_validation_pred,
            )
        ],
        "Bias (%)": [
            bias_pct(
                y_validation,
                xgb_validation_pred,
            )
        ],
    }
).set_index("Model")

xgb_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
XGBoost,124.18,191.63,8.81,2.51


In [19]:
validation_results = pd.concat(
    [
        baseline_results,
        rf_results,
        xgb_results,
    ]
)

validation_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Seasonal Naive (7-day),170.05,298.87,12.07,0.18
Rolling Mean (28-day),232.46,364.12,16.50,-0.69
Random Forest,111.52,178.77,7.91,0.50
XGBoost,124.18,191.63,8.81,2.51


### XGBoost Hyperparameter Tuning

A small set of XGBoost configurations is evaluated on the validation period. The search is intentionally constrained to avoid excessive tuning to a single validation window while assessing whether changes in model complexity and learning rate improve generalization.

In [20]:
xgb_configs = [
    {
        "name": "XGB-1",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 4,
    },
    {
        "name": "XGB-2",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 4,
    },
    {
        "name": "XGB-3",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 5,
    },
    {
        "name": "XGB-4",
        "n_estimators": 700,
        "learning_rate": 0.03,
        "max_depth": 5,
    },
]

xgb_tuning_results = []
xgb_candidates = {}

for config in xgb_configs:
    model = build_xgboost_pipeline(
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        random_state=42,
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    xgb_candidates[config["name"]] = model

    xgb_tuning_results.append(
        {
            "Model": config["name"],
            "Trees": config["n_estimators"],
            "Learning Rate": config["learning_rate"],
            "Max Depth": config["max_depth"],
            "MAE": mae(y_validation, predictions),
            "RMSE": rmse(y_validation, predictions),
            "WAPE (%)": wape(y_validation, predictions),
            "Bias (%)": bias_pct(y_validation, predictions),
        }
    )

xgb_tuning_results = (
    pd.DataFrame(xgb_tuning_results)
    .set_index("Model")
    .sort_values("WAPE (%)")
)

xgb_tuning_results.round(2)

,Trees,Learning Rate,Max Depth,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,,,,
XGB-1,300,0.05,4,111.31,176.40,7.90,0.40
XGB-2,500,0.03,4,112.06,178.70,7.95,0.27
XGB-3,500,0.05,5,115.58,178.71,8.20,1.42
XGB-4,700,0.03,5,120.75,184.36,8.57,1.16


### XGBoost Tuning Results

Reducing XGBoost tree depth substantially improved validation performance compared with the initial configuration. The strongest configuration used 300 trees, a learning rate of 0.05, and maximum depth of 4, achieving a WAPE of **7.90%**, MAE of **111.31**, RMSE of **176.40**, and bias of **+0.40%**.

This performance is effectively comparable to Random Forest, which achieved a WAPE of 7.91%. Increasing model complexity did not improve validation performance; configurations using deeper trees produced higher forecast errors.

Given the very small difference between the strongest XGBoost configuration and Random Forest, validation results do not indicate a practically meaningful advantage for either model.

In [21]:
best_xgb = xgb_candidates["XGB-1"]

xgb_best_validation_pred = best_xgb.predict(
    X_validation
)

In [22]:
validation_analysis = validation[
    [
        "date",
        "store_id",
        "state_id",
        "cat_id",
        "sales",
    ]
].copy()

validation_analysis["rf_prediction"] = rf_validation_pred
validation_analysis["xgb_prediction"] = xgb_best_validation_pred

validation_analysis["rf_abs_error"] = (
    validation_analysis["sales"]
    - validation_analysis["rf_prediction"]
).abs()

validation_analysis["xgb_abs_error"] = (
    validation_analysis["sales"]
    - validation_analysis["xgb_prediction"]
).abs()

validation_analysis.head()

,date,store_id,state_id,cat_id,sales,rf_prediction,xgb_prediction,rf_abs_error,xgb_abs_error
1829,2016-03-28,CA_1,CA,FOODS,2480,2505.790000,2553.859375,25.790000,73.859375
1830,2016-03-29,CA_1,CA,FOODS,2481,2452.020000,2363.050049,28.980000,117.949951
1831,2016-03-30,CA_1,CA,FOODS,2135,2363.103333,2357.831299,228.103333,222.831299
1832,2016-03-31,CA_1,CA,FOODS,2651,2448.366667,2390.865967,202.633333,260.134033
1833,2016-04-01,CA_1,CA,FOODS,3132,3128.970000,3182.505127,3.030000,50.505127


In [23]:
category_results = []

for category, group in validation_analysis.groupby("cat_id"):
    category_results.append(
        {
            "Category": category,
            "RF WAPE (%)": wape(
                group["sales"],
                group["rf_prediction"],
            ),
            "XGB WAPE (%)": wape(
                group["sales"],
                group["xgb_prediction"],
            ),
        }
    )

category_results = (
    pd.DataFrame(category_results)
    .set_index("Category")
)

category_results.round(2)

,RF WAPE (%),XGB WAPE (%)
Category,,
FOODS,7.22,7.04
HOBBIES,12.57,13.14
HOUSEHOLD,8.04,8.24


In [24]:
state_results = []

for state, group in validation_analysis.groupby("state_id"):
    state_results.append(
        {
            "State": state,
            "RF WAPE (%)": wape(
                group["sales"],
                group["rf_prediction"],
            ),
            "XGB WAPE (%)": wape(
                group["sales"],
                group["xgb_prediction"],
            ),
        }
    )

state_results = (
    pd.DataFrame(state_results)
    .set_index("State")
)

state_results.round(2)

,RF WAPE (%),XGB WAPE (%)
State,,
CA,6.89,6.95
TX,8.22,8.18
WI,9.06,8.95


In [25]:
import importlib
import src.models.tune_models as tune_module

importlib.reload(tune_module)

cv_folds = tune_module.create_time_series_cv_folds(
    train,
    n_splits=4,
    horizon_days=28,
)

for fold_number, (cv_train, cv_validation) in enumerate(
    cv_folds,
    start=1,
):
    print(f"Fold {fold_number}")

    print(
        "  Train:",
        cv_train["date"].min().date(),
        "to",
        cv_train["date"].max().date(),
        "|",
        f"{len(cv_train):,} rows",
    )

    print(
        "  Validation:",
        cv_validation["date"].min().date(),
        "to",
        cv_validation["date"].max().date(),
        "|",
        f"{len(cv_validation):,} rows",
        "|",
        cv_validation["date"].nunique(),
        "days",
    )

    print()

Fold 1
  Train: 2011-03-26 to 2015-12-06 | 51,510 rows
  Validation: 2015-12-07 to 2016-01-03 | 840 rows | 28 days

Fold 2
  Train: 2011-03-26 to 2016-01-03 | 52,350 rows
  Validation: 2016-01-04 to 2016-01-31 | 840 rows | 28 days

Fold 3
  Train: 2011-03-26 to 2016-01-31 | 53,190 rows
  Validation: 2016-02-01 to 2016-02-28 | 840 rows | 28 days

Fold 4
  Train: 2011-03-26 to 2016-02-28 | 54,030 rows
  Validation: 2016-02-29 to 2016-03-27 | 840 rows | 28 days



In [26]:
import importlib
import src.models.tune_models as tune_module

importlib.reload(tune_module)

print(
    hasattr(
        tune_module,
        "evaluate_model_cv",
    )
)

True


In [27]:
from src.models.train_random_forest import (
    build_random_forest_pipeline,
)

rf_cv_results = tune_module.evaluate_model_cv(
    model_builder=build_random_forest_pipeline,
    model_params={
        "n_estimators": 300,
        "max_depth": None,
        "random_state": 42,
    },
    folds=cv_folds,
)

rf_cv_results["fold_results"].round(2)

,fold,mae,rmse,wape,bias
0,1,161.70,299.53,13.48,-1.25
1,2,179.09,288.55,13.76,-6.32
2,3,136.70,217.21,9.77,-3.32
3,4,133.92,227.90,9.57,0.86


## Hyperparameter Tuning

Random Forest and XGBoost are tuned using expanding-window
time-series cross-validation. Each validation fold contains
28 complete days to match the forecasting horizon.

In [28]:
import importlib

import src.models.tune_models as tune_module
import src.models.train_random_forest as rf_module

# Reload local modules so notebook uses the latest saved code.
importlib.reload(tune_module)
importlib.reload(rf_module)

<module 'src.models.train_random_forest' from 'c:\\Data Science\\GitHub\\retail-demand-forecasting\\src\\models\\train_random_forest.py'>

In [29]:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 15, 25],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.7],
    "random_state": [42],
}

In [30]:
rf_test_params = {
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_leaf": 2,
    "max_features": 0.7,
    "random_state": 42,
}

rf_test_cv = tune_module.evaluate_model_cv(
    model_builder=rf_module.build_random_forest_pipeline,
    model_params=rf_test_params,
    folds=cv_folds,
)

rf_test_cv["fold_results"].round(2)

,fold,mae,rmse,wape,bias
0,1,159.72,292.56,13.31,-0.77
1,2,172.05,279.67,13.22,-6.28
2,3,137.01,217.93,9.79,-4.25
3,4,134.41,231.18,9.60,0.21


In [31]:
print(f"Mean WAPE: {rf_test_cv['mean_wape']:.2f}%")
print(f"WAPE Std:  {rf_test_cv['std_wape']:.2f}%")
print(f"Mean MAE:   {rf_test_cv['mean_mae']:.2f}")
print(f"Mean RMSE:  {rf_test_cv['mean_rmse']:.2f}")
print(f"Mean Bias:  {rf_test_cv['mean_bias']:.2f}%")

Mean WAPE: 11.48%
WAPE Std:  2.06%
Mean MAE:   150.80
Mean RMSE:  255.33
Mean Bias:  -2.78%


In [32]:
importlib.reload(tune_module)

<module 'src.models.tune_models' from 'c:\\Data Science\\GitHub\\retail-demand-forecasting\\src\\models\\tune_models.py'>

In [33]:
rf_tuning_results = tune_module.search_hyperparameters(
    model_builder=rf_module.build_random_forest_pipeline,
    param_grid=rf_param_grid,
    folds=cv_folds,
)

Configuration 1/36: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42}
  Mean WAPE: 10.72%
Configuration 2/36: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 0.7, 'random_state': 42}
  Mean WAPE: 11.41%
Configuration 3/36: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'random_state': 42}
  Mean WAPE: 10.80%
Configuration 4/36: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 0.7, 'random_state': 42}
  Mean WAPE: 11.39%
Configuration 5/36: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'random_state': 42}
  Mean WAPE: 10.95%
Configuration 6/36: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 4, 'max_features': 0.7, 'random_state': 42}
  Mean WAPE: 11.48%
Configuration 7/36: {'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state':

In [34]:
rf_tuning_results.head(10).round(3)

,n_estimators,max_depth,min_samples_leaf,max_features,random_state,mean_wape,std_wape,mean_mae,mean_rmse,mean_bias
0,400,NaN,1,sqrt,42,10.661,1.475,140.252,243.643,-3.054
1,400,25.0,1,sqrt,42,10.669,1.567,140.296,244.148,-3.036
2,200,NaN,1,sqrt,42,10.716,1.533,140.943,245.606,-3.082
3,200,25.0,1,sqrt,42,10.723,1.600,140.994,244.565,-3.034
4,400,25.0,2,sqrt,42,10.752,1.540,141.420,247.281,-3.159
5,200,NaN,2,sqrt,42,10.797,1.581,141.987,249.414,-3.191
6,200,25.0,2,sqrt,42,10.805,1.545,142.117,248.124,-3.174
7,400,NaN,2,sqrt,42,10.806,1.573,142.110,248.862,-3.225
8,400,15.0,1,sqrt,42,10.845,1.579,142.625,247.787,-3.263
9,400,15.0,2,sqrt,42,10.863,1.613,142.835,249.850,-3.260


In [35]:
best_rf_params = (
    rf_tuning_results
    .iloc[0][
        [
            "n_estimators",
            "max_depth",
            "min_samples_leaf",
            "max_features",
            "random_state",
        ]
    ]
    .to_dict()
)

best_rf_params

{'n_estimators': 400,
 'max_depth': nan,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'random_state': 42}

In [43]:
best_rf_params = {
    "n_estimators": 400,
    "max_depth": None,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "random_state": 42,
}

In [44]:
print(best_rf_params)
print(type(best_rf_params["max_depth"]))

{'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42}
<class 'NoneType'>


### Random Forest Tuning Results

Random Forest hyperparameters were evaluated using four expanding-window
cross-validation folds, each with a 28-day forecast horizon.

The selected configuration used 400 trees, unrestricted tree depth,
a minimum leaf size of 1, and square-root feature sampling. It achieved
a mean cross-validation WAPE of **10.66%**, compared with approximately
**11.65%** for the initial Random Forest configuration.

The standard deviation of WAPE across folds also decreased from
approximately **2.29** to **1.48 percentage points**, indicating more
consistent performance across forecast periods.

The tuned model exhibited an average bias of **-3.05%**, indicating
a modest tendency to underforecast across the cross-validation periods.

In [36]:
importlib.reload(tune_module)

import src.models.train_xgboost as xgb_module
importlib.reload(xgb_module)

<module 'src.models.train_xgboost' from 'c:\\Data Science\\GitHub\\retail-demand-forecasting\\src\\models\\train_xgboost.py'>

In [37]:
xgb_param_grid = {
    "n_estimators": [200, 300, 500],
    "learning_rate": [0.03, 0.05, 0.08],
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [1.0, 5.0],
    "random_state": [42],
}

In [38]:
xgb_tuning_results = (
    tune_module.randomized_hyperparameter_search(
        model_builder=xgb_module.build_xgboost_pipeline,
        param_grid=xgb_param_grid,
        folds=cv_folds,
        n_iter=12,
        random_state=42,
    )
)

Configuration 1/12: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42}
  Mean WAPE: 11.05%
Configuration 2/12: {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 5.0, 'random_state': 42}
  Mean WAPE: 11.48%
Configuration 3/12: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 5.0, 'random_state': 42}
  Mean WAPE: 10.71%
Configuration 4/12: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 5.0, 'random_state': 42}
  Mean WAPE: 11.09%
Configuration 5/12: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 3, 

In [39]:
xgb_tuning_results.head(10).round(3)

,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,random_state,mean_wape,std_wape,mean_mae,mean_rmse,mean_bias
0,500,0.05,5,1,1.0,1.0,0.0,1.0,42,9.871,0.682,130.437,205.606,-0.833
1,500,0.08,4,1,1.0,0.8,0.0,5.0,42,9.950,0.891,131.259,209.147,-0.607
2,200,0.05,5,5,1.0,1.0,0.0,5.0,42,10.502,0.946,138.623,220.493,-1.051
3,300,0.03,5,5,0.8,0.8,0.1,5.0,42,10.710,1.298,141.232,226.401,-1.395
4,200,0.05,4,3,0.8,0.8,0.0,5.0,42,11.022,1.430,145.274,232.502,-1.357
5,200,0.05,4,5,0.8,1.0,0.0,1.0,42,11.051,1.398,145.699,233.584,-1.593
6,300,0.03,4,3,0.8,1.0,0.0,5.0,42,11.087,1.454,146.056,234.281,-1.452
7,500,0.03,3,1,0.8,0.8,0.0,1.0,42,11.420,1.793,150.375,242.392,-1.340
8,200,0.03,4,1,0.8,0.8,0.1,5.0,42,11.476,1.893,150.970,244.772,-1.871
9,200,0.03,4,3,0.8,0.8,0.0,5.0,42,11.517,1.905,151.482,246.383,-1.831


In [40]:
best_xgb_params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
}

### Cross-Validation Model Selection

Hyperparameters were evaluated using four expanding-window
cross-validation folds, each representing a 28-day forecasting horizon.

The tuned Random Forest achieved a mean CV WAPE of **10.66%**
with a standard deviation of **1.48 percentage points**.

The selected XGBoost configuration achieved a lower mean CV WAPE
of **9.87%** and a lower standard deviation of **0.68 percentage
points**. It also produced lower mean MAE and RMSE and substantially
lower average forecast bias.

Based on cross-validation performance, XGBoost was selected as the
leading candidate for evaluation on the separately held-out validation
period.

## Tuned Model Validation

The selected Random Forest and XGBoost configurations are retrained
using the full training period and evaluated on the separately held-out
28-day validation period.

This validation period was not used during hyperparameter tuning and
therefore provides an independent comparison of the tuned models before
final model selection.

In [45]:
tuned_rf = rf_module.build_random_forest_pipeline(
    **best_rf_params
)

tuned_rf.fit(
    X_train,
    y_train,
)

tuned_rf_validation_pred = tuned_rf.predict(
    X_validation
)

In [46]:
tuned_xgb = xgb_module.build_xgboost_pipeline(
    **best_xgb_params
)

tuned_xgb.fit(
    X_train,
    y_train,
)

tuned_xgb_validation_pred = tuned_xgb.predict(
    X_validation
)

In [47]:
tuned_validation_results = pd.DataFrame(
    [
        {
            "Model": "Tuned Random Forest",
            "MAE": mae(
                y_validation,
                tuned_rf_validation_pred,
            ),
            "RMSE": rmse(
                y_validation,
                tuned_rf_validation_pred,
            ),
            "WAPE (%)": wape(
                y_validation,
                tuned_rf_validation_pred,
            ),
            "Bias (%)": bias_pct(
                y_validation,
                tuned_rf_validation_pred,
            ),
        },
        {
            "Model": "Tuned XGBoost",
            "MAE": mae(
                y_validation,
                tuned_xgb_validation_pred,
            ),
            "RMSE": rmse(
                y_validation,
                tuned_xgb_validation_pred,
            ),
            "WAPE (%)": wape(
                y_validation,
                tuned_xgb_validation_pred,
            ),
            "Bias (%)": bias_pct(
                y_validation,
                tuned_xgb_validation_pred,
            ),
        },
    ]
).set_index("Model")

tuned_validation_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Tuned Random Forest,108.24,180.88,7.68,-1.13
Tuned XGBoost,115.58,178.71,8.20,1.42


### Tuned Model Validation Results

The model ranking differed between cross-validation and the separately
held-out validation period.

Across the four historical cross-validation windows, XGBoost achieved
the stronger average performance and greater stability. On the final
28-day validation period, however, Random Forest achieved the lower
WAPE (**7.68% vs. 8.20%**) and MAE (**108.24 vs. 115.58**), while
XGBoost achieved a slightly lower RMSE (**178.71 vs. 180.88**).

These results indicate that relative model performance varies across
forecast periods. No additional hyperparameter tuning was performed
using the final validation period in order to preserve its role as an
independent model-selection checkpoint.

In [48]:
strict_baseline_pred = seasonal_naive_forecast(
    validation,
    lag_days=28,
)

strict_baseline_results = pd.DataFrame(
    [
        {
            "Model": "Seasonal Naive (28-day)",
            "MAE": mae(
                y_validation,
                strict_baseline_pred,
            ),
            "RMSE": rmse(
                y_validation,
                strict_baseline_pred,
            ),
            "WAPE (%)": wape(
                y_validation,
                strict_baseline_pred,
            ),
            "Bias (%)": bias_pct(
                y_validation,
                strict_baseline_pred,
            ),
        }
    ]
).set_index("Model")

strict_baseline_results.round(2)

,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Seasonal Naive (28-day),149.24,255.32,10.59,-0.64


## Final Model Selection

A 28-day seasonal-naive forecast was used as the horizon-safe benchmark,
ensuring that all predictions rely only on demand observed before the
forecast origin.

On the held-out validation period, the seasonal baseline achieved a
WAPE of **10.59%**. The tuned Random Forest reduced WAPE to **7.68%**,
while the tuned XGBoost model achieved **8.20%**.

Although XGBoost achieved stronger average performance across the
historical cross-validation windows, Random Forest achieved the lowest
WAPE and MAE on the separately held-out validation period. Random
Forest was therefore selected for final evaluation.

No further model or hyperparameter changes were made after this
selection to avoid adapting the modeling process to the final test set.

## Final Test Evaluation

After model selection, the selected Random Forest configuration is
retrained using the combined training and validation history. The
resulting model is evaluated once on the untouched final 28-day test
period.

In [49]:
train_validation = pd.concat(
    [
        train,
        validation,
    ],
    ignore_index=True,
)

X_train_validation, y_train_validation = prepare_ml_data(
    train_validation
)

X_test, y_test = prepare_ml_data(test)

print(
    "Training through:",
    train_validation["date"].max().date(),
)

print(
    "Test period:",
    test["date"].min().date(),
    "to",
    test["date"].max().date(),
)

print(
    "Training rows:",
    f"{len(X_train_validation):,}",
)

print(
    "Test rows:",
    f"{len(X_test):,}",
)

Training through: 2016-04-24
Test period: 2016-04-25 to 2016-05-22
Training rows: 55,710
Test rows: 840
